# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# One row:
#   One row represents a single unique content_id (i.e. page on a website) on a specific date
#   within a month with it's search and engagement metrics calculated for a day.


# Grain:
#   One row per (content_hash_id × report_date)


# Tables Used:
#   Primary table -> fact_content_daily_performance (partitioned by month)
#   Secondary tables (context only) -> dim_content, dim_clients


# Time Window:
#   Development window -> month = '2026-03'
#   Rolling window per row -> Each row shows that specific day's metrics.


# Predict/Rank (label/proxy):
#   This is unsupervised learning. There is no pre-existing label.
#   We have 5 numerical features (impressions, clicks, CTR, position, engagement).
#   K-Means will assign each content_id to one of 4 clusters.
#   Cluster membership is the outcome, created AFTER training.


# Data to deliberately exclude:
#   Query text (anonymized hashes) -> salted pseudonyms that can't be reversed to recover text
#   Rows with zero impressions -> Can't characterize page performance
#   Rows with missing clicks data -> Incomplete activity data
#   Brand and navigational queries (if identifiable) -> Behave differently from informational queries
#   Days with insufficient data -> Very new content may cluster artificially

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# Features:
#   1. gsc_impressions -> Daily impressions in Google Search Console
#   2. gsc_clicks	-> Daily clicks from search results
#   3. gsc_avg_position	-> Average daily SERP ranking position
#   4. ga4_pageviews -> Daily pageviews after organic click
#   5. ga4_total_engagement_sec -> Total engagement time per day
#   6. scroll_events -> Daily scroll interactions on page


# Context:
#   1. report_date -> Date of this daily snapshot
#   2. client_hash_id	-> Hashed client identifier
#   3. content_hash_id -> Hashed content identifier
#   4. month -> Calendar month (YYYY-MM)
#   5. gsc_data_available	-> Boolean: GSC data available this day
#   6. ga4_data_available	-> Boolean: GA4 data available this day


# Label:
#   None - unsupervised clustering


# Excluded:
#   1. client_has_gsc
#      Why: Redundant quality flag; use gsc_data_available instead

#   2. client_has_ga4
#      Why: Redundant quality flag; use ga4_data_available instead

#   3. gsc_sum_position
#      Why: Should use avg_position, not sum (sum is meaningless for positions)

#   4. ga4_sessions
#      Why: Overlaps with ga4_engaged_sessions; redundant

#   5. ga4_users
#      Why: Counts unique users; not a direct engagement proxy

#   6. sessions_direct
#      Why: Traffic source detail; adds noise (we care about content quality, not source)

#   7. sessions_referral
#      Why: Traffic source detail; adds noise

#   8. sessions_social
#      Why: Traffic source detail; adds noise

#   9. sessions_paid
#      Why: Traffic source detail; adds noise

#  10. ai_chatgpt
#      Why: AI source breakdowns are too granular; use sessions_ai aggregate instead

#  11. ai_perplexity
#      Why: AI source breakdowns are too granular

#  12. ai_gemini
#      Why: AI source breakdowns are too granular

#  13. ai_copilot
#      Why: AI source breakdowns are too granular

#  14. ai_claude
#      Why: AI source breakdowns are too granular

#  15. ai_meta
#      Why: AI source breakdowns are too granular

#  16. ai_other
#      Why: AI source breakdowns are too granular

#  17. sessions_organic
#      Why: Counts organic sessions; overlaps with impressions/clicks

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

print("Connected to HuggingFace warehouse via DuckDB")

Connected to HuggingFace warehouse via DuckDB


In [18]:
# QUERY 1: Verify grain (one row per content per date)

result1 = con.sql("""
  SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT content_hash_id) as unique_content_ids,
    COUNT(DISTINCT report_date) as unique_dates,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT content_hash_id), 2) as avg_rows_per_content
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
  WHERE month = '2026-03'
""")

print("QUERY 1: GRAIN VERIFICATION")
print("-"*70)
df1 = result1.df()
print(df1.to_string(index=False))

total = df1['total_rows'][0]
unique_content = df1['unique_content_ids'][0]
unique_dates = df1['unique_dates'][0]
avg_rows = df1['avg_rows_per_content'][0]

print("\n" + "-"*70)
print(f"  Grain verified: {avg_rows} rows per content_id (expected ~31 for March)")
print(f"  Total rows: {total:,}")
print(f"  Unique content_ids: {unique_content:,}")
print(f"  Unique dates: {unique_dates}")
print(f"  Grain: one row per (content_id × date)")
print("-"*70)

QUERY 1: GRAIN VERIFICATION
----------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  unique_content_ids  unique_dates  avg_rows_per_content
    9841378              331437            31                 29.69

----------------------------------------------------------------------
  Grain verified: 29.69 rows per content_id (expected ~31 for March)
  Total rows: 9,841,378
  Unique content_ids: 331,437
  Unique dates: 31
  Grain: one row per (content_id × date)
----------------------------------------------------------------------


In [20]:
# QUERY 2: Row count and date span for a slice

result2 = con.sql("""
  SELECT
    COUNT(*) as row_count,
    MIN(report_date) as earliest_date,
    MAX(report_date) as latest_date,
    COUNT(DISTINCT report_date) as num_days
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
  WHERE month = '2026-03'
""")

print("QUERY 2: ROW COUNT & DATE SPAN")
print("-"*70)
df2 = result2.df()
print(df2.to_string(index=False))

row_count = df2['row_count'][0]
earliest = df2['earliest_date'][0]
latest = df2['latest_date'][0]
num_days = df2['num_days'][0]

print("\n" + "-"*70)
print("INTERPRETATION:")
print(f"  Data slice verified:")
print(f"  Row count: {row_count:,} total rows")
print(f"  Date range: {earliest} to {latest}")
print(f"  Number of days: {num_days}")
print(f"  Time window: One calendar month")
print("-"*70)

QUERY 2: ROW COUNT & DATE SPAN
----------------------------------------------------------------------
 row_count earliest_date latest_date  num_days
   9841378    2026-03-01  2026-03-31        31

----------------------------------------------------------------------
INTERPRETATION:
  Data slice verified:
  Row count: 9,841,378 total rows
  Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
  Number of days: 31
  Time window: One calendar month
----------------------------------------------------------------------


In [21]:
# QUERY 3: Availability of quality data

result3 = con.sql("""
  SELECT
    COUNT(*) as total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available = TRUE) as rows_gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available = TRUE) as rows_ga4_available,
    COUNT(*) FILTER (WHERE gsc_data_available = TRUE AND ga4_data_available = TRUE) as rows_both_available,
    COUNT(*) FILTER (WHERE gsc_impressions > 0) as rows_with_impressions,
    COUNT(*) FILTER (WHERE gsc_impressions > 0 AND gsc_clicks >= 0) as rows_with_activity,
    ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_impressions > 0 AND gsc_clicks >= 0) / COUNT(*), 2) as pct_with_activity
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
  WHERE month = '2026-03'
""")

print("QUERY 3: AVAILABILITY / QUALITY FILTER")
print("-"*70)
df3 = result3.df()
print(df3.to_string(index=False))

total = df3['total_rows'][0]
gsc_avail = df3['rows_gsc_available'][0]
ga4_avail = df3['rows_ga4_available'][0]
both_avail = df3['rows_both_available'][0]
with_activity = df3['rows_with_activity'][0]
pct_activity = df3['pct_with_activity'][0]

print("\n" + "-"*70)
print("INTERPRETATION:")
print(f"  Data availability verified:")
print(f"  Total rows: {total:,}")
print(f"  Rows with GSC data available: {gsc_avail:,} ({100*gsc_avail/total:.1f}%)")
print(f"  Rows with GA4 data available: {ga4_avail:,} ({100*ga4_avail/total:.1f}%)")
print(f"  Rows with both GSC & GA4: {both_avail:,} ({100*both_avail/total:.1f}%)")
print(f"  Rows with impressions > 0 AND clicks >= 0: {with_activity:,} ({pct_activity}%)")
print(f"\n  → {pct_activity}% of rows have usable activity data")
print("-"*70)

QUERY 3: AVAILABILITY / QUALITY FILTER
----------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  rows_gsc_available  rows_ga4_available  rows_both_available  rows_with_impressions  rows_with_activity  pct_with_activity
    9841378             3611061              413966               364347                3611061             3611061              36.69

----------------------------------------------------------------------
INTERPRETATION:
  Data availability verified:
  Total rows: 9,841,378
  Rows with GSC data available: 3,611,061 (36.7%)
  Rows with GA4 data available: 413,966 (4.2%)
  Rows with both GSC & GA4: 364,347 (3.7%)
  Rows with impressions > 0 AND clicks >= 0: 3,611,061 (36.69%)

  → 36.69% of rows have usable activity data
----------------------------------------------------------------------


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.